### WIYN Open Cluster Study. XCVII. An Extended Radial-velocity Survey and Spectroscopic Binary Orbits in the Open Cluster NGC 188

Ritvik Sai Narayan, Evan Linck, Robert D. Mathieu, and Aaron M. Geller

https://iopscience.iop.org/article/10.3847/1538-3881/ae2d14


They provide 35 new spectroscopic-binary orbits from their (RV) survey of the old (6.4 ± 0.2 Gyr) open cluster NGC 188. 

They also find several BSS systems 

"We find that NGC 188 has 18 secure BSSs:
WOCS 2679, 4230, 4290, 4306, 4348, 4535, 4581, 4589, 4970,
5078, 5325, 5350, 5379, 5434, 5467, 5885, 5934, and 8104.
Four other stars in the BSS region of the color–magnitude
diagram (CMD), WOCS 4447, 4540, 4945, and 5020, "

This notebook is to crossmatch those systems with the new orbital solutions in their Table 6, and then update systems in our catalog if needed. 

In [9]:
import numpy as np
import json
import pandas as pd

import astropy.units as u
from astroquery.vizier import Vizier
import re

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

In [8]:
BSS_IDS = [2679, 4230, 4290, 4306, 4348, 4535, 4581, 4589, 4970,
5078, 5325, 5350, 5379, 5434, 5467, 5885, 5934, 8104]
BSS_Candidates = [4447, 4540, 4945, 5020]


In [14]:
col_names = [
    "PKM", "Per", "e_Per", "Ncyc", "gamma", "e_gamma", "K", "e_K",
    "e", "e_e", "omega", "e_omega", "T0", "e_T0", "asini",
    "e_asini", "fm", "e_fm", "sig", "Nobs"
 ]

table_6 = pd.read_csv(DATA_DIR / "from_others" / "R_Narayan26.txt",
    sep=r"\s+",engine="python", skiprows=30,names=col_names,)

# WOCS IDs are integers;
table_6["PKM"] = table_6["PKM"].astype(int)

In [18]:
display(table_6.head())
print(f"len {len(table_6)}")

,PKM,Per,e_Per,Ncyc,gamma,e_gamma,K,e_K,e,e_e,omega,e_omega,T0,e_T0,asini,e_asini,fm,e_fm,sig,Nobs
0,269,553.00000,3.00000,11.3,-42.10,0.40,7.10,0.90,0.720,0.050,69.0,12.0,55313.00,17.00,38.00,6.00,0.0070,0.0030,0.886,15
1,583,119.78000,0.04000,55.1,-41.22,0.14,6.27,0.22,0.180,0.040,64.0,12.0,55474.00,4.00,10.20,0.40,0.0029,0.0003,0.618,20
2,3719,51.57450,0.00160,128.6,-43.06,0.22,28.10,0.40,0.562,0.008,225.8,1.4,56022.17,0.12,16.46,0.24,0.0670,0.0030,0.959,23
3,3755,10.32601,0.00009,642.2,-42.60,0.30,39.10,0.40,0.206,0.011,72.0,3.0,55602.72,0.08,5.43,0.06,0.0598,0.0018,1.083,19
4,3953,940.40000,2.50000,7.1,-41.14,0.14,4.51,0.19,0.040,0.040,240.0,70.0,56340.00,170.00,58.20,2.50,0.0089,0.0011,0.633,24


len 35


In [19]:
targets_secure = set(BSS_IDS)
targets_candidates = set(BSS_Candidates)
targets_all = targets_secure | targets_candidates

matches = table_6[table_6["PKM"].isin(targets_all)].copy().sort_values("PKM")
matches["is_secure_BSS"] = matches["PKM"].isin(targets_secure)
matches["is_candidate"] = matches["PKM"].isin(targets_candidates)

display(matches[["PKM", "Per", "e", "fm", "Nobs", "is_secure_BSS", "is_candidate"]])
print(f"Matched IDs: {matches['PKM'].tolist()}")


# missing_secure = sorted(targets_secure - set(matches["PKM"]))
# missing_candidates = sorted(targets_candidates - set(matches["PKM"]))
# print(f"Missing secure BSS IDs: {missing_secure}")
# print(f"Missing candidate IDs: {missing_candidates}")

,PKM,Per,e,fm,Nobs,is_secure_BSS,is_candidate
10,4230,0.455609,0.04,0.0039,32,True,False
18,4945,5110.000000,0.48,0.1400,15,False,True
19,5020,38000.000000,0.74,0.0400,27,False,True
32,8104,3460.000000,0.12,0.0520,25,True,False


Matched IDs: [4230, 4945, 5020, 8104]


### Next we wan't to create a json table for these systems, to be added to the master table.